In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.animation as animation
import matplotlib.cm as cm
import json
from tqdm import tqdm
from PIL import Image
import os
import gc
import warnings
%matplotlib inline
warnings.filterwarnings('ignore')

# Tomographic Reconstruction of Basketball Movement Data - Pacers @ Raptors, October 28th, 2015

In this case study, we will be using SportsVU movement data alongside play by play data to construct meaningful visualizations of The Toronto Raptors offense. The analysis below details the data cleaning, preprocessing, visualization and machine learning used to produce spatial reconstruction using ball movement data. The end goal is to reconstruct offensive threat given the ball's position on the court using naive models (conditional probability, movement heatmaps) and interpretable machine learning models outlined in a study conducted by Jairo Diaz-Rodriguez and his peers. This notebook aims to be an extension of the methods used in the study (link below).

Code and analysis completed by Jonathan Channer under the supervision of Dr. Jairo Diaz-Rodriguez.

Play by play data: [Kaggle](https://www.kaggle.com/datasets/brains14482/nba-playbyplay-and-shotdetails-data-19962021?select=nbastats_2015.csv)  
Play by play parsing: [Ryan Davis](https://github.com/rd11490/NBA_Tutorials/tree/master/play_by_play_parser)   
SportsVU game logs: [Kostya Linou](https://github.com/linouk23/NBA-Player-Movements/tree/master)  
Tomographic reconstruction of a disease transmission landscape via GPS recorded random paths: [Article](https://arxiv.org/pdf/2404.04455)

## Reading the Data

The play-by-play parser provides possession segments for both teams. Inspection of this game showed that the parser's `possession_start` is not a reliable movement boundary: for many rows it is equal to that row's end time. The corrected interval therefore starts at the preceding **game segment's** end, applied before filtering to Toronto. The raw parser times and original game-row number are retained for auditability.

`SOURCE_POSSESSION_NUMBER` is assigned once from Toronto's source order and is never renumbered. After converting the game-elapsed values to quarter-clock values, any interval whose start equals its end is retained in possession metadata and point totals but flagged as `zero_duration`. These stopped-clock records can represent valid play-by-play segments, including free throws; this notebook does not infer their cause or merge them automatically. Because a zero-length interval cannot support a movement path, it is also flagged `movement_eligible = False` and excluded from spatial models without fabricating a duration or borrowing a nearby tracking frame.

In [ ]:
df = pd.read_csv("Data/0021500009_possessions.csv")

In [ ]:
df

In [ ]:
# Preserve the parser values before correcting interval starts.
df = df.copy()
df["source_row_number"] = np.arange(1, len(df) + 1)
df["parser_possession_start"] = df["possession_start"]
df["parser_possession_end"] = df["possession_end"]

# The usable start is the preceding game segment's end. Do this before team filtering.
df["possession_start"] = df["parser_possession_end"].shift(1)
first_segment_mask = df["possession_start"].isna()
df.loc[first_segment_mask, "possession_start"] = df.loc[
    first_segment_mask, "parser_possession_start"
]
df.head()

In [ ]:
df_raptors = df.loc[df["possession_team"] == 1610612761].copy()
df_raptors["SOURCE_POSSESSION_NUMBER"] = np.arange(1, len(df_raptors) + 1)

In [ ]:
df_raptors.head()

In [ ]:
print(sum(df["team1_points"]), " - " ,sum(df["team2_points"]))

In [ ]:
times_score = df_raptors[[
    "source_row_number",
    "SOURCE_POSSESSION_NUMBER",
    "period",
    "parser_possession_start",
    "parser_possession_end",
    "possession_start",
    "possession_end",
    "team2_points",
]].copy()

# Keep the original analysis column as a stable alias for downstream cells.
times_score["possession_number"] = times_score["SOURCE_POSSESSION_NUMBER"]

In [ ]:
times_score

In [ ]:
# Convert game-elapsed boundaries to the quarter clock used by SportsVU.
updated_times_score = times_score.copy()
period_end_elapsed = updated_times_score["period"] * 720
for boundary in ["possession_start", "possession_end"]:
    updated_times_score[boundary] = period_end_elapsed - updated_times_score[boundary]

updated_times_score["possession_duration_seconds"] = (
    updated_times_score["possession_start"] - updated_times_score["possession_end"]
)
if (updated_times_score["possession_duration_seconds"] < 0).any():
    raise ValueError("Corrected possession intervals contain a negative duration.")

updated_times_score["zero_duration"] = np.isclose(
    updated_times_score["possession_duration_seconds"], 0
)
updated_times_score["movement_eligible"] = (
    ~updated_times_score["zero_duration"]
    & (updated_times_score["possession_duration_seconds"] > 0)
)
updated_times_score["scored"] = (updated_times_score["team2_points"] > 0).astype(int)

if updated_times_score["SOURCE_POSSESSION_NUMBER"].duplicated().any():
    raise ValueError("SOURCE_POSSESSION_NUMBER must remain unique.")

stopped_clock_possessions = updated_times_score.loc[
    updated_times_score["zero_duration"],
    ["SOURCE_POSSESSION_NUMBER", "period", "possession_start", "team2_points"],
].copy()
print(f"Raptors possession metadata rows: {len(updated_times_score)}")
print(f"Movement-eligible possessions: {updated_times_score['movement_eligible'].sum()}")
print(f"Stopped-clock possessions retained in metadata: {len(stopped_clock_possessions)}")
display(stopped_clock_possessions)

In [ ]:
updated_times_score

In [ ]:
updated_times_score.tail(10)

# TO DO
# parse JSON file for positional data of the ball in the time interval of possesion_start and possession_end and create a matrix with it and the outcome (scored points)
# Maybe find a way to determine if the ball is a shot using the z-axis to clean up the output 

In [ ]:
sum(updated_times_score["scored"])/len(updated_times_score)

## Parsing the Movement Data

SportsVU contains many samples, including repeated quarter-clock timestamps around stoppages. Movement is extracted only for positive-duration rows (`movement_eligible`); zero-duration rows remain in `updated_times_score` and all scoring totals but are intentionally absent from movement models. No nearest frame is substituted. Repeated timestamps are reduced with an explicit `tail(1)` within each possession, period, and time, so one possession cannot remove another possession's boundary sample.

The validation below compares the complete 103-possession metadata table with movement coverage. It reports excluded and unexpectedly missing source IDs and their points, then fails if any movement-eligible possession is absent. This notebook deliberately does not classify turnovers, rebounds, or any other defensive cause. Existing files under `Data/` remain products of their prior run until this notebook is rerun from the top.

In [ ]:
file = open("0021500009.json")

data = json.load(file)

print(data["quarters"]["1"][0][5][0])

print(data["quarters"]["1"][0][2])

print(data["home"]["teamid"])
file.close()

del data
gc.collect()

# data["quarters"]["quarter_num"][moment][index 2: time] -> comparison for possesion time interval
# data["quarters"]["quarter_num"][moment][index 5: ball/player positional info][index 0: ball data][indices 2, 3, 4: x, y and z positions respectively] -> position data to be written to feature matrix

In [ ]:
# Load JSON data only once
with open("0021500009.json", "r") as f:
    tracking_data = json.load(f)

# Preload quarters once (avoid repeated dict lookup)
quarters = tracking_data.get("quarters", {})

# Store filtered ball data
ball_pos_data = []

# Zero-duration rows stay in metadata/point totals but cannot define a movement path.
movement_metadata = updated_times_score.loc[
    updated_times_score["movement_eligible"]
].copy()

for _, row in tqdm(
    movement_metadata.iterrows(),
    total=len(movement_metadata),
    desc="Processing movement-eligible possessions",
):
    period = str(int(row["period"]))
    start = row["possession_start"]
    end = row["possession_end"]
    source_possession = int(row["SOURCE_POSSESSION_NUMBER"])
    scored = int(row["scored"])
    points_scored = int(row["team2_points"])

    # Skip if no data for that period
    moments = quarters.get(period, [])
    if not moments:
        continue

    # Use list comprehension for speed
    filtered_moments = [
        {
            "period": int(period),
            "time": moment[2],
            "x": moment[5][0][2],
            "y": moment[5][0][3],
            "z": moment[5][0][4],
            "possession_start": start,
            "possession_end": end,
            "source_row_number": int(row["source_row_number"]),
            "SOURCE_POSSESSION_NUMBER": source_possession,
            "possession_number": source_possession,
            "points_scored": points_scored,
            "scored": scored,
            "movement_eligible": True,
            "zero_duration": False,
        }
        for moment in moments
        if isinstance(moment, list) and
           len(moment) > 5 and
           len(moment[5]) > 0 and
           end <= moment[2] <= start
    ]

    ball_pos_data.extend(filtered_moments)

ball_pos_df = pd.DataFrame(ball_pos_data)
print(
    f"Filtered {len(ball_pos_df)} moments across "
    f"{ball_pos_df['SOURCE_POSSESSION_NUMBER'].nunique()} possessions."
)

print(ball_pos_df.shape)
print(ball_pos_df.columns)
print(ball_pos_df.head())
ball_pos_df.to_pickle("Data/ball_pos_df.pkl")

# Clean up memory
del tracking_data, quarters, ball_pos_data
gc.collect()

In [ ]:
ball_pos_df = pd.read_pickle("Data/ball_pos_df.pkl")
required_tracking_columns = {
    "SOURCE_POSSESSION_NUMBER",
    "possession_number",
    "points_scored",
    "movement_eligible",
    "zero_duration",
}
missing_tracking_columns = required_tracking_columns.difference(ball_pos_df.columns)
if missing_tracking_columns:
    raise ValueError(
        "The cached movement pickle predates the interval fix. "
        f"Re-run the extraction cell; missing columns: {sorted(missing_tracking_columns)}"
    )

In [ ]:
ball_pos_df.describe()

In [ ]:
# Keep exactly one sample per repeated clock value within each possession.
ball_pos_df["row_index"] = ball_pos_df.index

last_rows_in_order = (
    ball_pos_df.groupby(
        ["possession_number", "period", "time"],
        group_keys=False,
        sort=False,
    )
    .tail(1)
    .sort_values("row_index")
    .drop(columns="row_index")
    .reset_index(drop=True)
)

metadata_ids = set(updated_times_score["SOURCE_POSSESSION_NUMBER"].astype(int))
eligible_metadata = updated_times_score.loc[
    updated_times_score["movement_eligible"]
]
eligible_ids = set(eligible_metadata["SOURCE_POSSESSION_NUMBER"].astype(int))
movement_ids = set(last_rows_in_order["SOURCE_POSSESSION_NUMBER"].astype(int))

excluded_metadata = updated_times_score.loc[
    ~updated_times_score["movement_eligible"]
]
missing_metadata_ids = sorted(metadata_ids - movement_ids)
missing_eligible_ids = sorted(eligible_ids - movement_ids)
unexpected_movement_ids = sorted(movement_ids - eligible_ids)
missing_metadata_points = int(
    updated_times_score.loc[
        updated_times_score["SOURCE_POSSESSION_NUMBER"].isin(missing_metadata_ids),
        "team2_points",
    ].sum()
)
missing_eligible_points = int(
    eligible_metadata.loc[
        eligible_metadata["SOURCE_POSSESSION_NUMBER"].isin(missing_eligible_ids),
        "team2_points",
    ].sum()
)

movement_coverage_report = pd.Series({
    "metadata_possessions": len(updated_times_score),
    "metadata_points": int(updated_times_score["team2_points"].sum()),
    "movement_eligible_possessions": len(eligible_metadata),
    "movement_eligible_points": int(eligible_metadata["team2_points"].sum()),
    "movement_possessions_found": len(movement_ids),
    "excluded_zero_duration_ids": sorted(
        excluded_metadata["SOURCE_POSSESSION_NUMBER"].astype(int).tolist()
    ),
    "excluded_zero_duration_points": int(excluded_metadata["team2_points"].sum()),
    "all_metadata_ids_missing_from_movement": missing_metadata_ids,
    "all_metadata_points_missing_from_movement": missing_metadata_points,
    "missing_eligible_ids": missing_eligible_ids,
    "missing_eligible_points": missing_eligible_points,
    "unexpected_movement_ids": unexpected_movement_ids,
}, name="value").to_frame()
display(movement_coverage_report)

if missing_eligible_ids or unexpected_movement_ids:
    raise ValueError(
        "Movement coverage failed: every positive-duration source possession "
        "must be represented exactly within the eligible ID set."
    )

In [ ]:
last_rows_in_order.to_csv("Data/final.csv")
last_rows_in_order.tail(50)
# Possession is a tuple for some reason. Coverting to appropriate dtypes.

In [ ]:
last_rows_in_order["SOURCE_POSSESSION_NUMBER"] = last_rows_in_order["SOURCE_POSSESSION_NUMBER"].astype(int)
last_rows_in_order["possession_number"] = last_rows_in_order["possession_number"].astype(int)
last_rows_in_order["scored"] = last_rows_in_order["scored"].astype(int)

last_rows_in_order.describe()

In [ ]:
# We now have our movement dataframe. From here we will:
# 1. Create an adjacency matrix of the court using numpy and PIL
# 2. Vectorize our movement data
# 3. Create our GLM with sigmoid as the input into a -log inverse minimization problem with total variation to apply smoothness
# 4. Map the output to our court
# 5. Troubleshoot and optimize

In [ ]:
# Heatmap by possession -> Naive visualization
# How to discretize -> Start small, get bigger (proportional court sizes, 10x20)
# Adjacency matrix for movement data (count of movement through each square during a possession)
# Create our GLM with sigmoid (uh oh) as the input into a -log inverse minimization problem with total variation to apply smoothness (Augmentation? Bootstrapping?)
# Confidence intervals after resampling
# Naive output -> split x-matrix into target groups (y = 1, y = 0), calculate probability score per square
# Per quarter analysis

In [ ]:
last_rows_in_order.isna().sum()

In [ ]:
last_rows_in_order.to_pickle("Data/cleaned_df.pkl")

## Ball Movement Visualizations

Now we begin with visualizing our data. Because we are using basketball data, we want to map our movements directly onto a court image using the `Image`, `numpy`, and `matplotlib` packages. Using `Image` and `matplotlib` is fairly straightforward, but using `numpy` required careful planning and statistical knowledge. Below is an outline of some of the mathematics used to discretize our court into an adjacency matrix and map ball movements to said matrix.

### Discretizing the Court

To begin, we needed to create a discrete map of our court. An NBA basketball court has the dimensions 94×50 feet, totaling 4700 sq ft. A matrix of this resolution is too computationally expensive to work with, so we reduce it to 200 units, or a 20×10 bin grid. What this creates is a set of discrete, uniform squares on a grid, which we will call $\mathcal{M}$, where each position $(i, j)$ corresponds to a grid cell:

$$
\mathcal{M} = \left\{ (i, j) \;\middle|\; 0 \leq i < 20,\; 0 \leq j < 10 \right\}
$$

With this grid, we map our movement data on the court to each position on the grid using the following transformation:

$$
\forall\; (x, y) \in \mathcal{D},\quad
x_{\text{bin}} = \min\left( \left\lfloor \frac{x}{L} \cdot N_x \right\rfloor,\; N_x - 1 \right), \quad
y_{\text{bin}} = \min\left( \left\lfloor \frac{y}{W} \cdot N_y \right\rfloor,\; N_y - 1 \right)
$$

Where:
- $\mathcal{D}$ is the dataset of continuous $(x, y)$ coordinates,
- $L$ and $W$ are the court length and width (94 and 50 feet),
- $N_x$ and $N_y$ are the number of bins in the horizontal and vertical directions,
- $\lfloor \cdot \rfloor$ is the floor function.

This transformation maps continuous positions into discrete bins in $\mathcal{M}$, aligning each data point with a grid cell.

We then construct a heatmap of ball movement, denoted by $\mathcal{H}$, and apply normalization to scale all values between 0 and 1:

$$
\mathcal{H}_{\text{norm}}(i, j) = \frac{\mathcal{H}(i, j)}{\max\limits_{(i, j) \in \mathcal{M}} \mathcal{H}(i, j)}
$$

This results in a normalized density heatmap where each grid position in $\mathcal{M}$ is associated with a score on the interval $(0, 1)$, representing the relative frequency of ball presence.


In [ ]:
court_img = Image.open("court.jpg")

court_length = 94  # NBA court length in feet
court_width = 50   # NBA court width in feet

x_bins = 20  # number of horizontal bins
y_bins = 10  # number of vertical bins

# --- Discretize x and y positions into grid indices ---
last_rows_in_order['x_bin'] = np.clip((last_rows_in_order['x'] / court_length * x_bins).astype(int), 0, x_bins - 1)
last_rows_in_order['y_bin'] = np.clip((last_rows_in_order['y'] / court_width * y_bins).astype(int), 0, y_bins - 1)

# --- Create Heatmap Matrix ---
heatmap = np.zeros((x_bins, y_bins))
for _, row in last_rows_in_order.iterrows():
    heatmap[row['x_bin'], row['y_bin']] += 1

# Normalize (optional, for better visualization)
heatmap_normalized = heatmap / np.max(heatmap)

# --- Plot ---
fig, ax = plt.subplots(figsize=(12, 6))

# Show court image behind the heatmap
ax.imshow(court_img, extent=[0, x_bins, 0, y_bins], aspect='auto')

# Show heatmap overlay
heatmap_img = ax.imshow(
    heatmap_normalized.T,
    cmap='hot',
    origin='lower',
    extent=[0, x_bins, 0, y_bins],
    alpha=0.6  # transparency
)

# Add labels and colorbar
plt.colorbar(heatmap_img, ax=ax, label='Normalized Ball Movement Density')
ax.set_title("Ball Movement Heatmap Over NBA Court (20×10 Grid)")
ax.set_xlabel("Court Length Bins (20)")
ax.set_ylabel("Court Width Bins (10)")
plt.tight_layout()
plt.show()

In [ ]:
# First plot above
# Rotate points 180 degrees if their y-axis is above 47 feet

last_rows_in_order = last_rows_in_order.drop(columns=["x_bin", "y_bin"])

In [ ]:
movement_df = last_rows_in_order.copy()

mask = movement_df['x'] >= 47

movement_df.loc[mask, 'x'] = 94 - movement_df.loc[mask, 'x']
movement_df.loc[mask, 'y'] = 50 - movement_df.loc[mask, 'y']
# This is likely incorrrect...

In [ ]:
movement_df.describe()

In [ ]:
def heat(df, img, title = "Ball Movement Heatmap"):
    
    court_img = Image.open(img)

    court_length = 94  # NBA court length in feet (reduced)
    court_width = 50   # NBA court width in feet

    x_bins = 20  # number of horizontal bins
    y_bins = 10  # number of vertical bins
    
    # --- Discretize x and y positions into grid indices ---
    df['x_bin'] = np.clip((df['x'] / court_length * x_bins).astype(int), 0, x_bins - 1)
    df['y_bin'] = np.clip((df['y'] / court_width * y_bins).astype(int), 0, y_bins - 1)
    
    # --- Create Heatmap Matrix ---
    heatmap = np.zeros((x_bins, y_bins))
    for _, row in df.iterrows():
        heatmap[row['x_bin'], row['y_bin']] += 1
    
    # Normalize (optional, for better visualization)
    heatmap_normalized = heatmap / np.max(heatmap)
    
    # --- Plot ---
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Show court image behind the heatmap
    ax.imshow(court_img, extent=[0, x_bins, 0, y_bins], aspect='auto')
    
    # Show heatmap overlay
    heatmap_img = ax.imshow(
    heatmap_normalized.T,
    cmap='hot',
    origin='lower',
    extent=[0, x_bins, 0, y_bins],
    alpha=0.6  # transparency
    )
    
    # Add labels and colorbar
    plt.colorbar(heatmap_img, ax=ax, label='Normalized Ball Movement Density')
    ax.set_title(title)
    ax.set_xlabel("Court Length Bins (20)")
    ax.set_ylabel("Court Width Bins (10)")
    plt.tight_layout()
    plt.show()

heat(last_rows_in_order, "court.jpg", )

In [ ]:
heat(last_rows_in_order[last_rows_in_order["scored"] == 1], "court.jpg", "Ball Movement Heatmap on Scoring Plays")

In [ ]:
heat(last_rows_in_order[last_rows_in_order["scored"] == 0], "court.jpg", "Ball Movement Heatmap on Non-Scoring Plays")
# Why does the ball seem to have more movement on non-scoring plays?
# Less efficient movement?
# Maybe filter on the z-axis to remove some noise from the basket area?

## Conditional Scoring Probability by Sector

We estimate the conditional probability that a possession results in a score **given that the ball passed through a specific sector** of the court.

Let $\mathcal{M} = \left\{ (i, j) \mid 0 \leq i < N_x,\; 0 \leq j < N_y \right\}$ be the set of spatial bins on the court.

For each possession $p$, we define:
- $\mathcal{V}_p \subseteq \mathcal{M}$ as the set of all bins visited during possession $p$
- $s_p \in \{0, 1\}$ as an indicator whether the possession resulted in a score

We define two matrices:
- $T(i, j)$: total number of possessions that entered bin $(i, j)$
- $S(i, j)$: number of **scored** possessions that entered bin $(i, j)$

Then the estimated conditional scoring probability is:

$$
P(\text{score} \mid (i, j)) = 
\begin{cases}
\dfrac{S(i, j)}{T(i, j)} & \text{if } T(i, j) > 0 \\
0 & \text{otherwise}
\end{cases}
$$

This is implemented in code by iterating over all possessions, marking visited bins, and updating counts for $T(i, j)$ and $S(i, j)$. The resulting probability matrix is visualized as a heatmap overlayed on a court image.

We set the dimensions of the court to 20x10 for easy comparison to our ball movement heatmaps and included a filter in a second method to disregard any sectors that had less than $\mathcal{n}$ possesions move through it where $n \in \mathbb{N}$.

In [ ]:
def calculate_score_probability_matrix(df, x_bins=20, y_bins=10):

    # Bin positions
    df['x_bin'] = np.clip((df['x'] / 94 * x_bins).astype(int), 0, x_bins - 1)
    df['y_bin'] = np.clip((df['y'] / 50 * y_bins).astype(int), 0, y_bins - 1)

    # Initialize per-bin possession counters
    total_possessions = np.zeros((x_bins, y_bins))
    scored_possessions = np.zeros((x_bins, y_bins))

    # Count unique possessions that entered each bin
    grouped = df.groupby("possession_number")
    for _, group in grouped:
        visited = set(zip(group['x_bin'], group['y_bin']))
        scored = group['scored'].iloc[0]  # same for all rows in a possession
        for xb, yb in visited:
            total_possessions[xb, yb] += 1
            if scored == 1:
                scored_possessions[xb, yb] += 1

    # Compute probability matrix
    with np.errstate(divide='ignore', invalid='ignore'):
        probability_matrix = np.true_divide(scored_possessions, total_possessions)
        probability_matrix[~np.isfinite(probability_matrix)] = 0  # Set NaNs and infs to 0

    return probability_matrix

prob_matrix = calculate_score_probability_matrix(last_rows_in_order)

court_img = Image.open("court.jpg")

fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(court_img, extent=[0, 20, 0, 10], aspect='auto')
overlay = ax.imshow(
    prob_matrix.T,
    cmap='viridis',
    origin='lower',
    extent=[0, 20, 0, 10],
    alpha=0.6
)

plt.colorbar(overlay, ax=ax, label='Probabilty')
ax.set_title("Ball Movement Scoring Probability (Noisy)")
ax.set_xlabel("Court Length Bins (20)")
ax.set_ylabel("Court Width Bins (10)")
plt.tight_layout()
plt.show()

In [ ]:
def calculate_score_probability_matrix_filter(df, min_visits, x_bins=20, y_bins=10):

    # Bin positions
    df['x_bin'] = np.clip((df['x'] / 94 * x_bins).astype(int), 0, x_bins - 1)
    df['y_bin'] = np.clip((df['y'] / 50 * y_bins).astype(int), 0, y_bins - 1)

    # Initialize per-bin possession counters
    total_possessions = np.zeros((x_bins, y_bins))
    scored_possessions = np.zeros((x_bins, y_bins))

    # Count unique possessions that entered each bin
    for _, group in df.groupby("possession_number"):
        visited_bins = set(zip(group['x_bin'], group['y_bin']))
        scored = group['scored'].iloc[0]  # same for all rows in a possession
        for xb, yb in visited_bins:
            total_possessions[xb, yb] += 1
            if scored == 1:
                scored_possessions[xb, yb] += 1

    # Calculate conditional probabilities
    with np.errstate(divide='ignore', invalid='ignore'):
        prob_matrix = scored_possessions / total_possessions
        prob_matrix[~np.isfinite(prob_matrix)] = 0

    # Filter out bins with too few possessions
    prob_matrix[total_possessions < min_visits] = np.nan

    return prob_matrix

min_visits = 5

prob_matrix = calculate_score_probability_matrix_filter(last_rows_in_order, min_visits, x_bins=20, y_bins=10)
court_img = Image.open("court.jpg")

fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(court_img, extent=[0, 20, 0, 10], aspect='auto')
overlay = ax.imshow(
    prob_matrix.T,
    cmap='viridis',
    origin='lower',
    extent=[0, 20, 0, 10],
    alpha=0.6
)

plt.colorbar(overlay, ax=ax, label='Probabilty')
ax.set_title(f"Ball Movement Scoring Probability (Filtered ≥{min_visits} Possessions)")
ax.set_xlabel("Court Length Bins (20)")
ax.set_ylabel("Court Width Bins (10)")
plt.tight_layout()
plt.show()

In [ ]:
# High probability towards the boundaries. Inbound passes?
# Probability is higher than expected in most zones. Filter z-axis? 

## Plotting Possessions to Improve Analysis

The heatmaps are useful, but we can add more context to them before doing breakdown analysis by analyzing plays visually. This will hopefully clear up the picture as to why the ball moves more during non-scoring plays, and as to why plays that move along the bottom side of the court are so effective.

This will also allow us to see how noisy some of our data is, or if it's even tracked correctly.

In [ ]:
movement_df.to_csv("Data/movement_csv.csv")

In [ ]:
def movement_plot_possession(df, possession, image):
    court_img = mpimg.imread(image)
    
    # Filter a specific possession
    sub_df = df[df["possession_number"] == possession].sort_values(by="time", ascending=False)
    
    # Get start and end positions
    start_x, start_y = sub_df.iloc[0][["x", "y"]]
    end_x, end_y = sub_df.iloc[-1][["x", "y"]]
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.imshow(court_img, extent=[0, 94, 0, 50])
    ax.plot(sub_df["x"], sub_df["y"], color="blue", linewidth=2, marker="o", markersize=3, label="Path")
    
    # Start marker
    ax.plot(start_x, start_y, marker="s", color="green", markersize=10, label="Start")
    
    # End marker
    ax.plot(end_x, end_y, marker="X", color="red", markersize=10, label="End")
    
    # Styling
    ax.set_xlim(0, 104)
    ax.set_ylim(0, 50)
    ax.set_title(f"Ball Movement: Possession {possession} with Start and End")
    ax.set_xlabel("Court X")
    ax.set_ylabel("Court Y")
    ax.legend()
    
    plt.show()

movement_plot_possession(last_rows_in_order, 23,"court.jpg")
# 20, 29

In [ ]:
def movement_plot_all_quarters(df, image):
    court_img = mpimg.imread(image)
    fig, axes = plt.subplots(4, 2, figsize=(18, 24))

    # Define fixed court image extent
    court_extent = [0, 94, 0, 50]

    # Compute axis limits based on data, allowing buffer
    x_min, x_max = min(-5, df["x"].min()), max(100, df["x"].max())
    y_min, y_max = min(-5, df["y"].min()), max(51, df["y"].max())

    for row, quarter in enumerate([1, 2, 3, 4]):
        for col, scored in enumerate([0, 1]):
            ax = axes[row, col]
            scored_df = df[(df["period"] == quarter) & (df["scored"] == scored)]
            possession_ids = scored_df["possession_number"].unique()
            possession_count = len(possession_ids)
            cmap = cm.get_cmap('tab20', max(possession_count, 1))

            # Show fixed-size court image
            ax.imshow(court_img, extent=court_extent, zorder=0)

            # Extend the axes beyond court image
            ax.set_xlim(x_min, x_max)
            ax.set_ylim(y_min, y_max)
            ax.set_title(f"Quarter {quarter} | Scored = {bool(scored)} (n = {possession_count})")
            ax.set_xlabel("Court X")
            ax.set_ylabel("Court Y")

            for i, pid in enumerate(possession_ids):
                group = scored_df[scored_df["possession_number"] == pid].sort_values(by="time", ascending=False)
                start_x, start_y = group.iloc[0][["x", "y"]]
                end_x, end_y = group.iloc[-1][["x", "y"]]
                color = cmap(i)
                ax.plot(group["x"], group["y"], color=color, linewidth=2, alpha=0.9, zorder=1)
                ax.plot(start_x, start_y, marker="s", color="blue", markersize=8, zorder=2)
                ax.plot(end_x, end_y, marker="X", color="red", markersize=8, zorder=2)

    plt.tight_layout()
    plt.show()

# Usage
movement_plot_all_quarters(last_rows_in_order, "court.jpg")

In [ ]:
last_rows_in_order.describe()

In [ ]:
movement_plot_all_quarters(last_rows_in_order[last_rows_in_order["z"] < 10], "court.jpg")

In [ ]:
def heat(df, img, ax, title="Ball Movement Heatmap"):
    court_img = Image.open(img)

    court_length = 94  # NBA court length in feet
    court_width = 50   # NBA court width in feet

    x_bins = 20
    y_bins = 10

    # Bin the coordinates
    df['x_bin'] = np.clip((df['x'] / court_length * x_bins).astype(int), 0, x_bins - 1)
    df['y_bin'] = np.clip((df['y'] / court_width * y_bins).astype(int), 0, y_bins - 1)

    # Create heatmap
    heatmap = np.zeros((x_bins, y_bins))
    for _, row in df.iterrows():
        heatmap[row['x_bin'], row['y_bin']] += 1

    # Normalize
    heatmap_normalized = heatmap / np.max(heatmap) if np.max(heatmap) > 0 else heatmap

    # Plot background court
    ax.imshow(court_img, extent=[0, x_bins, 0, y_bins], aspect='auto')

    # Overlay heatmap
    im = ax.imshow(
        heatmap_normalized.T,
        cmap='hot',
        origin='lower',
        extent=[0, x_bins, 0, y_bins],
        alpha=0.6
    )

    ax.set_title(title)
    ax.set_xlabel("Court Length Bins (20)")
    ax.set_ylabel("Court Width Bins (10)")
    
    return im  # Return image object for colorbar

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Ball Movement Heatmap on Non-Scoring Plays (z < 12)", fontsize=18)

# Shared image and heatmap ranges
court_image_path = "court.jpg"
vmin, vmax = 0, 1  # consistent color scale if needed

# Track one of the returned im objects for colorbar
heatmap_images = []

for i, period in enumerate(range(1, 5)):
    ax = axes[i // 2, i % 2]
    subset = last_rows_in_order[
        (last_rows_in_order["scored"] == 0) &
        #(last_rows_in_order["z"] < 12) &
        (last_rows_in_order["period"] == period)
    ]
    im = heat(subset, court_image_path, ax, f"Period {period} (n={len(subset)})")
    heatmap_images.append(im)

# Add one shared colorbar (optional)
plt.tight_layout(rect=[0, 0.03, 0.90, 0.95])  # leave room for colorbar
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])  # colorbar axes
fig.colorbar(heatmap_images[0], cax=cbar_ax, label="Normalized Ball Movement Density")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Ball Movement Heatmap on Scoring Plays (z < 12)", fontsize=18)

# Shared image and heatmap ranges
court_image_path = "court.jpg"
vmin, vmax = 0, 1  # consistent color scale if needed

# Track one of the returned im objects for colorbar
heatmap_images = []

for i, period in enumerate(range(1, 5)):
    ax = axes[i // 2, i % 2]
    subset = last_rows_in_order[
        (last_rows_in_order["scored"] == 1) &
        #(last_rows_in_order["z"] < 12) &
        (last_rows_in_order["period"] == period)
    ]
    im = heat(subset, court_image_path, ax, f"Period {period} (n={len(subset)})")
    heatmap_images.append(im)

# Add one shared colorbar (optional)
plt.tight_layout(rect=[0, 0.03, 0.90, 0.95])  # leave room for colorbar
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])  # colorbar axes
fig.colorbar(heatmap_images[0], cax=cbar_ax, label="Normalized Ball Movement Density")
plt.show()

In [ ]:
min_visits = 3
x_bins = 20
y_bins = 10
court_img = Image.open("court.jpg")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(f"Scoring Probability Maps by Period (≥{min_visits} Possessions)", fontsize=18)

prob_maps = []  # to store overlay handles for shared colorbar

for i, period in enumerate(range(1, 5)):
    ax = axes[i // 2, i % 2]
    subset = last_rows_in_order[
        #(last_rows_in_order["z"] < 12) &
        (last_rows_in_order["period"] == period)
    ]

    prob_matrix = calculate_score_probability_matrix_filter(subset, min_visits, x_bins, y_bins)

    # Plot court image
    ax.imshow(court_img, extent=[0, x_bins, 0, y_bins], aspect='auto')

    # Overlay probability heatmap
    overlay = ax.imshow(
        prob_matrix.T,
        cmap='viridis',
        origin='lower',
        extent=[0, x_bins, 0, y_bins],
        alpha=0.6,
        vmin=0, vmax=1  # for consistent color scale across plots
    )

    ax.set_title(f"Period {period}")
    ax.set_xlabel("Court Length Bins")
    ax.set_ylabel("Court Width Bins")
    prob_maps.append(overlay)

# Add shared colorbar without overlapping subplots
plt.tight_layout(rect=[0, 0.03, 0.90, 0.95])
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
fig.colorbar(prob_maps[0], cax=cbar_ax, label="Scoring Probability")

plt.show()

In [ ]:
inverted = last_rows_in_order.copy()

mask = inverted["period"] > 2

inverted.loc[mask, "x_bin"] = 20 - inverted.loc[mask, "x_bin"]
inverted.loc[mask, "y_bin"] = 10 - inverted.loc[mask, "y_bin"]

In [ ]:
def heat_invert(df, img, ax, title="Ball Movement Heatmap"):
    court_img = Image.open(img)

    court_length = 94
    court_width = 50
    x_bins, y_bins = 20, 10

    # Bin coordinates
    df['x_bin'] = np.clip((df['x'] / court_length * x_bins).astype(int), 0, x_bins - 1)
    df['y_bin'] = np.clip((df['y'] / court_width * y_bins).astype(int), 0, y_bins - 1)

    # Flip bins for possessions going the other direction
    mask = df["period"] > 2
    df.loc[mask, "x_bin"] = x_bins - 1 - df.loc[mask, "x_bin"]
    df.loc[mask, "y_bin"] = y_bins - 1 - df.loc[mask, "y_bin"]

    df['x_bin'] = np.clip(df['x_bin'], 0, x_bins - 1)
    df['y_bin'] = np.clip(df['y_bin'], 0, y_bins - 1)

    # Heatmap computation
    heatmap = np.zeros((x_bins, y_bins))
    np.add.at(heatmap, (df['x_bin'], df['y_bin']), 1)

    # Normalize
    heatmap_normalized = heatmap / np.max(heatmap) if np.max(heatmap) > 0 else heatmap

    # Plotting
    ax.imshow(court_img, extent=[0, x_bins, 0, y_bins], aspect='auto')
    im = ax.imshow(
        heatmap_normalized.T,
        cmap='hot',
        origin='lower',
        extent=[0, x_bins, 0, y_bins],
        alpha=0.6
    )
    ax.set_title(title)
    ax.set_xlabel("Court Length Bins (20)")
    ax.set_ylabel("Court Width Bins (10)")
    
    return im

In [ ]:
def calculate_score_probability_matrix_filter_inverted(df, min_visits, x_bins=20, y_bins=10):

    # Bin positions
    df['x_bin'] = np.clip((df['x'] / 94 * x_bins).astype(int), 0, x_bins - 1)
    df['y_bin'] = np.clip((df['y'] / 50 * y_bins).astype(int), 0, y_bins - 1)

    df['x_bin'] = np.clip((df['x'] / court_length * x_bins).astype(int), 0, x_bins - 1)
    df['y_bin'] = np.clip((df['y'] / court_width * y_bins).astype(int), 0, y_bins - 1)

    mask = df["period"] > 2
    df.loc[mask, "x_bin"] = x_bins - 1 - df.loc[mask, "x_bin"]
    df.loc[mask, "y_bin"] = y_bins - 1 - df.loc[mask, "y_bin"]

    df['x_bin'] = np.clip(df['x_bin'], 0, x_bins - 1)
    df['y_bin'] = np.clip(df['y_bin'], 0, y_bins - 1)

    # Initialize per-bin possession counters
    total_possessions = np.zeros((x_bins, y_bins))
    scored_possessions = np.zeros((x_bins, y_bins))

    # Count unique possessions that entered each bin
    for _, group in df.groupby("possession_number"):
        visited_bins = set(zip(group['x_bin'], group['y_bin']))
        scored = group['scored'].iloc[0]  # same for all rows in a possession
        for xb, yb in visited_bins:
            total_possessions[xb, yb] += 1
            if scored == 1:
                scored_possessions[xb, yb] += 1

    # Calculate conditional probabilities
    with np.errstate(divide='ignore', invalid='ignore'):
        prob_matrix = scored_possessions / total_possessions
        prob_matrix[~np.isfinite(prob_matrix)] = 0

    # Filter out bins with too few possessions
    prob_matrix[total_possessions < min_visits] = np.nan

    return prob_matrix

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Inverted Ball Movement Heatmap on Non-Scoring Plays (z < 10)", fontsize=18)

# Shared image and heatmap ranges
court_image_path = "court.jpg"
vmin, vmax = 0, 1  # consistent color scale if needed

# Track one of the returned im objects for colorbar
heatmap_images = []

for i, period in enumerate(range(1, 5)):
    ax = axes[i // 2, i % 2]
    subset = last_rows_in_order[
        (last_rows_in_order["scored"] == 0) &
        #(last_rows_in_order["z"] < 10) &
        (last_rows_in_order["period"] == period)
    ]
    subset = subset[(subset["x"] >= 0) & (subset["x"] <= 94) &
                    (subset["y"] >= 0) & (subset["y"] <= 50)]

    if subset.empty:
        ax.axis("off")
        continue

    im = heat_invert(subset, court_image_path, ax, f"Period {period} (n={len(subset)})")
    heatmap_images.append(im)

# Add one shared colorbar (optional)
plt.tight_layout(rect=[0, 0.03, 0.90, 0.95])  # leave room for colorbar
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])  # colorbar axes
fig.colorbar(heatmap_images[0], cax=cbar_ax, label="Normalized Ball Movement Density")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Inverted Ball Movement Heatmap on Scoring Plays (z < 10)", fontsize=18)

# Shared image and heatmap ranges
court_image_path = "court.jpg"
vmin, vmax = 0, 1  # consistent color scale if needed

# Track one of the returned im objects for colorbar
heatmap_images = []

for i, period in enumerate(range(1, 5)):
    ax = axes[i // 2, i % 2]
    subset = last_rows_in_order[
        (last_rows_in_order["scored"] == 1) &
        #(last_rows_in_order["z"] < 10) &
        (last_rows_in_order["period"] == period)
    ]
    subset = subset[(subset["x"] >= 0) & (subset["x"] <= 94) &
                    (subset["y"] >= 0) & (subset["y"] <= 50)]

    if subset.empty:
        ax.axis("off")
        continue

    im = heat_invert(subset, court_image_path, ax, f"Period {period} (n={len(subset)})")
    heatmap_images.append(im)

# Add one shared colorbar (optional)
plt.tight_layout(rect=[0, 0.03, 0.90, 0.95])  # leave room for colorbar
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])  # colorbar axes
fig.colorbar(heatmap_images[0], cax=cbar_ax, label="Normalized Ball Movement Density")
plt.show()

In [ ]:
min_visits = 3
x_bins = 20
y_bins = 10
court_img = Image.open("court.jpg")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(f"Scoring Probability Maps by Period (≥{min_visits} Possessions)", fontsize=18)

prob_maps = []

for i, period in enumerate(range(1, 5)):
    ax = axes[i // 2, i % 2]
    subset = last_rows_in_order[
        (last_rows_in_order["z"] < 10) &
        (last_rows_in_order["period"] == period)
    ]

    prob_matrix = calculate_score_probability_matrix_filter_inverted(subset, min_visits, x_bins, y_bins)

    # Skip if matrix is all zeros or NaN
    if np.all(prob_matrix == 0) or np.isnan(prob_matrix).all():
        ax.axis("off")
        continue

    # Plot court and overlay
    ax.imshow(court_img, extent=[0, x_bins, 0, y_bins], aspect='auto')
    overlay = ax.imshow(
        prob_matrix.T,
        cmap='viridis',
        origin='lower',
        extent=[0, x_bins, 0, y_bins],
        alpha=0.6,
        vmin=0, vmax=1
    )

    ax.set_title(f"Period {period}")
    ax.set_xlabel("Court Length Bins")
    ax.set_ylabel("Court Width Bins")
    prob_maps.append(overlay)

# Only show colorbar if at least one heatmap was plotted
plt.tight_layout(rect=[0, 0.03, 0.90, 0.95])
if prob_maps:
    cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
    fig.colorbar(prob_maps[0], cax=cbar_ax, label="Scoring Probability")

plt.show()


## Conclusions Drawn from Naive Analysis and Visualizations

Above we have several plots that show direct movement, ball movement heatmaps per quarter and scoring probability maps. We inverted the heatmaps and probability maps for to interpret all offensive plays as going from left to right.

### Direct Movement

No conclusions can really be drawn from direct movement visualizations. However, this allows to see any errors in the tracking data or any erratic movements. There seems to be some errors within the measurement data, but this could easily be attributed to user error or tracking during timeouts. We combatted this as best as we could while cleaning the data and the results are still good. Overall, the ball movement seems to be in a good and that is shown in these visualizations.

### Ball Movement Heatmaps

The heatmaps give us much more to work with than the direct movement plots, so let's break them down. We set up some filters so we can see movement by `period`, `scored` and some `z` filtering. The first thing we notice is that ball movement appears to be much more erratic during non-scoring plays. This could be due to The Raptors players struggling to find scoring opportunities, forcing them to dribble or pass the ball more often which leads to a less consistent heatmap. Another finding is that we can see the ball was forced above the 3-point line during the first half of the game, indicating difficulty when attempting to drive to the basket or The Raptors were prioritizing 3-point attempts and failing them. Lastly, it appears that attacking on the left side of the basket was more effective, especially in the second half of the game. Their offense may have picked up on something and started to attack that side with a good amount of success. This is also shown in the probability maps that we will discuss next.

### Scoring Probability Maps

Similar to the movement heatmaps, we filtered them by quarter. We also filtered out any sectors that had less than three possesions move through them to handle any over/under representation of scoring in any given sector. Similar to the movement heatmaps, we can see the offensive struggles in the first half, especially in the first quarter. The Raptors were struggling to score early, with a moderate improvement in the second quarter. The second half tells a much different story with an offensive explosion. Both second half quarters show success when moving the ball to the right side of the basket, with solid success driving to the basket in the third quarter. We believe the biggest takeaway here is The Raptors were succesful in the second half, especially while attacking on the right side of the court. The third quarter appears to be the most succesful while attacking the basket when combining raw movement and the probability map, while quarter four shows more attempts at 3-point shooting.



In [ ]:
last_rows_in_order.to_csv("Data/analysis_ready.csv")

In [ ]:
last_rows_in_order.groupby("possession_number").size().sort_values(ascending=False)
# not many posessions have less than 100 samples
# consider this when bootstrapping

## Further Possession Cleaning

The below code blocks are to remove any possessions that cross the half court line twice or more. In NBA basketball games, this would be a half-court violation. There is the potential that not all the removed possessions are fouls, but could be an error with tracking, filtering or a plethora of other possible scenarios. Until a new solution is created, these possessions will simply be removed.

In [ ]:
# Drop possessions that cross half court twice or more
def count_half_court_crossings(possession_df, half_court_x=47):
    possession_df = possession_df.sort_values("time", ascending=False)
    side = np.sign(possession_df["x"] - half_court_x).replace(0, np.nan).ffill().bfill()
    if side.empty or side.isna().all():
        return 0
    return int(side.ne(side.shift()).sum() - 1)

half_court_crossings_df = (
    last_rows_in_order.groupby("possession_number")
    .apply(count_half_court_crossings)
    .rename("half_court_crossings")
    .reset_index()
)

possessions_to_drop = half_court_crossings_df.loc[
    half_court_crossings_df["half_court_crossings"] >= 2,
    "possession_number",
]

dropped_half_court_crossers_df = last_rows_in_order[
    last_rows_in_order["possession_number"].isin(possessions_to_drop)
].merge(half_court_crossings_df, on="possession_number", how="left")

last_rows_half_court_filtered = last_rows_in_order[
    ~last_rows_in_order["possession_number"].isin(possessions_to_drop)
].merge(half_court_crossings_df, on="possession_number", how="left")

print(f"Possessions before half-court filter: {last_rows_in_order['possession_number'].nunique()}")
print(f"Dropped possessions crossing half court twice or more: {len(possessions_to_drop)}")
print(f"Possessions after half-court filter: {last_rows_half_court_filtered['possession_number'].nunique()}")

display(last_rows_half_court_filtered)
display(dropped_half_court_crossers_df)

last_rows_half_court_filtered.to_csv("Data/analysis_ready_half_court_filtered.csv", index=False)


In [ ]:
dropped_half_court_crossers_df.groupby("possession_number")["scored"].first()
# 7 non-scoring, 2 scoring possessions